<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div><strong>Universidad Externado de Colombia</strong><br>
  <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
  <span>Docente: Wilmer Pineda-Ríos</span></div>
  <img src="../../assets/brand/logo-externado.png" width="190">
</div>

# Sesión 3 — Sensibilidad, Condorcet y puente a Bagging

**Objetivo.** Medir la inestabilidad de un árbol y explicar por qué agregar modelos puede reducirla.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
candidates = [
    Path("../../datasets/public/Bike_Sharing_Day.csv"),
    Path("datasets/public/Bike_Sharing_Day.csv"),
    Path("../datasets/public/Bike_Sharing_Day.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("No se encontró Bike_Sharing_Day.csv")

df = pd.read_csv(data_path, parse_dates=["dteday"]).sort_values("dteday").reset_index(drop=True)
df.head()

In [ ]:
target = "cnt"
leakage = ["casual", "registered"]
drop_columns = ["instant", "dteday", target, *leakage]
X = df.drop(columns=drop_columns)
y = df[target]

categorical = ["season", "mnth", "weekday", "weathersit"]
numeric = [c for c in X.columns if c not in categorical]

cut = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

preprocess = ColumnTransformer(
    [("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    remainder="passthrough",
)
cv = TimeSeriesSplit(n_splits=5)
X_train.shape, X_test.shape, (df.loc[cut, "dteday"], df["dteday"].max())

In [ ]:
def metrics(name, y_true, y_pred):
    return {
        "modelo": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

## 1. Pequeñas perturbaciones, árboles distintos

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
rows, preds = [], []
for b in range(40):
    idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    model = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(max_depth=6, min_samples_leaf=8, random_state=b))])
    model.fit(X_train.iloc[idx], y_train.iloc[idx])
    pred = model.predict(X_test)
    preds.append(pred)
    rows.append({"muestra": b, "MAE": mean_absolute_error(y_test, pred), "pred_primer_dia": pred[0]})
stability = pd.DataFrame(rows)
stability.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(stability["MAE"], bins=10, color="#0f766e", edgecolor="white")
axes[0].set(title="Variación del MAE", xlabel="MAE")
axes[1].hist(stability["pred_primer_dia"], bins=10, color="#c8962d", edgecolor="white")
axes[1].set(title="Variación de una predicción", xlabel="bicicletas")
plt.show()

## 2. Promediar predicciones

In [ ]:
prediction_matrix = np.vstack(preds)
single_mae = mean_absolute_error(y_test, prediction_matrix[0])
average_pred = prediction_matrix.mean(axis=0)
average_mae = mean_absolute_error(y_test, average_pred)
pd.DataFrame({"estrategia": ["un árbol", "promedio de 40"], "MAE": [single_mae, average_mae]}).round(2)

La reducción depende de la diversidad. Si todos los árboles cometen el mismo error, el promedio no puede corregirlo.

## 3. Condorcet como simulación

Suponemos clasificadores con probabilidad individual de acierto $p$ e independencia aproximada.

In [ ]:
from scipy.stats import binom

def majority_accuracy(n_models, p):
    threshold = n_models // 2 + 1
    return binom.sf(threshold - 1, n_models, p)

grid = pd.DataFrame(
    [(n, p, majority_accuracy(n, p)) for p in [0.45, 0.52, 0.60, 0.70] for n in range(1, 102, 2)],
    columns=["modelos", "p_individual", "p_mayoria"],
)
fig, ax = plt.subplots(figsize=(9, 4))
for p, group in grid.groupby("p_individual"):
    ax.plot(group["modelos"], group["p_mayoria"], label=f"p={p:.2f}")
ax.axhline(.5, color="#172029", linewidth=1)
ax.set(xlabel="número de votantes", ylabel="probabilidad de mayoría correcta", title="La mayoría ayuda solo bajo condiciones")
ax.legend()
plt.show()

## 4. La correlación pone un piso

Para árboles con varianza $\sigma^2$ y correlación media $\rho$:

$$\operatorname{Var}(\bar f)\approx \rho\sigma^2+\frac{1-\rho}{B}\sigma^2$$

In [ ]:
B = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(9, 4))
for rho in [0.0, 0.1, 0.4, 0.8]:
    relative_variance = rho + (1-rho)/B
    ax.plot(B, relative_variance, label=f"rho={rho}")
ax.set(xlabel="número de árboles B", ylabel="varianza relativa", title="Más árboles no eliminan errores correlacionados")
ax.legend()
plt.show()

## 5. Puente a Bagging

Bagging formaliza tres pasos: remuestrear con reemplazo, entrenar un estimador por muestra y agregar predicciones. Random Forest añadirá aleatoriedad de variables para reducir correlación.

## 6. Salida

1. ¿Qué significa que un árbol sea inestable?
2. ¿Por qué promediar puede reducir varianza?
3. ¿Qué supuestos necesita Condorcet?
4. ¿Qué papel cumple la correlación entre árboles?